# NB_bishop_ch13_figures
## Key Figures for Bishop Chapter 13 -- Graph Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_13/NB_bishop_ch13_figures.ipynb)

Generate publication-quality figures for graph structure, message passing, graph pooling, and spectral graph analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 13.1 -- Graph Structure

Example graph with labeled nodes carrying feature vectors, edges drawn explicitly, and the corresponding adjacency matrix shown alongside.

In [ ]:
# Define a small graph: 6 nodes with edges
node_labels = ['$v_1$', '$v_2$', '$v_3$', '$v_4$', '$v_5$', '$v_6$']
node_pos = {0: (0.0, 1.0), 1: (1.0, 2.0), 2: (2.0, 1.0),
            3: (1.0, 0.0), 4: (3.0, 2.0), 5: (3.0, 0.0)}
edges = [(0, 1), (0, 3), (1, 2), (1, 4), (2, 3), (2, 4), (2, 5), (3, 5)]
n_nodes = len(node_labels)

# Build adjacency matrix
A = np.zeros((n_nodes, n_nodes), dtype=int)
for i, j in edges:
    A[i, j] = 1
    A[j, i] = 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.4, 1]})

# Left panel: graph drawing
ax = axes[0]
ax.set_xlim(-0.8, 3.8)
ax.set_ylim(-0.8, 2.8)
ax.axis('off')
ax.set_title('Graph with Node Features', fontweight='bold', fontsize=14)

# Draw edges
for i, j in edges:
    xi, yi = node_pos[i]
    xj, yj = node_pos[j]
    ax.plot([xi, xj], [yi, yj], color='#888888', linewidth=1.8, zorder=1)

# Draw nodes
node_colors_list = [COLORS['blue'], COLORS['red'], COLORS['green'],
                    COLORS['amber'], COLORS['blue'], COLORS['red']]
for idx in range(n_nodes):
    x, y = node_pos[idx]
    circle = plt.Circle((x, y), 0.22, facecolor=node_colors_list[idx],
                         edgecolor='black', linewidth=1.5, zorder=3, alpha=0.85)
    ax.add_patch(circle)
    ax.text(x, y, node_labels[idx], ha='center', va='center',
            fontsize=13, fontweight='bold', color='white', zorder=4)
    # Feature vector annotation
    feat = np.random.randn(3).round(1)
    feat_str = f'[{feat[0]:.1f}, {feat[1]:.1f}, {feat[2]:.1f}]'
    offset_y = 0.35 if y > 0.5 else -0.35
    ax.text(x, y + offset_y, feat_str, ha='center', va='center',
            fontsize=7, color=node_colors_list[idx],
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                      edgecolor=node_colors_list[idx], alpha=0.8))

# Right panel: adjacency matrix
ax2 = axes[1]
im = ax2.imshow(A, cmap='Blues', vmin=0, vmax=1, aspect='equal')
ax2.set_xticks(range(n_nodes))
ax2.set_yticks(range(n_nodes))
short_labels = [f'$v_{i+1}$' for i in range(n_nodes)]
ax2.set_xticklabels(short_labels, fontsize=11)
ax2.set_yticklabels(short_labels, fontsize=11)
ax2.set_title('Adjacency Matrix $\\mathbf{A}$', fontweight='bold', fontsize=14)
for i in range(n_nodes):
    for j in range(n_nodes):
        color = 'white' if A[i, j] == 1 else 'black'
        ax2.text(j, i, str(A[i, j]), ha='center', va='center',
                 fontsize=11, fontweight='bold', color=color)

fig.tight_layout()
save_fig(fig, 'fig13_1_graph_structure')
plt.show()

## Figure 13.2 -- Message Passing GNN

GNN message passing mechanism: a target node aggregates messages from its neighbors, then applies an update function to produce a new embedding.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-1, 13)
ax.set_ylim(-1, 8)
ax.axis('off')

# --- Left side: graph with messages flowing ---
# Neighbor nodes
nb_pos = {0: (1.0, 6.5), 1: (0.0, 4.0), 2: (2.0, 4.0), 3: (0.0, 1.5)}
nb_labels = ['$v_1$', '$v_2$', '$v_3$', '$v_4$']
nb_colors = [COLORS['blue'], COLORS['green'], COLORS['amber'], COLORS['red']]

# Target node
target_pos = (4.5, 4.0)

# Draw neighbor nodes and message arrows
for idx, (pos, label, col) in enumerate(zip(nb_pos.values(), nb_labels, nb_colors)):
    circle = plt.Circle(pos, 0.35, facecolor=col, edgecolor='black',
                         linewidth=1.5, zorder=3, alpha=0.85)
    ax.add_patch(circle)
    ax.text(pos[0], pos[1], label, ha='center', va='center',
            fontsize=13, fontweight='bold', color='white', zorder=4)
    # Message arrow to target
    dx = target_pos[0] - pos[0]
    dy = target_pos[1] - pos[1]
    length = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / length, dy / length
    ax.annotate('', xy=(target_pos[0] - ux * 0.4, target_pos[1] - uy * 0.4),
                xytext=(pos[0] + ux * 0.4, pos[1] + uy * 0.4),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5, alpha=0.7))
    # Message label
    mid_x = (pos[0] + target_pos[0]) / 2 - 0.3
    mid_y = (pos[1] + target_pos[1]) / 2 + 0.2
    ax.text(mid_x, mid_y, f'$m_{{{idx+1}}}$', fontsize=10, color=col,
            fontweight='bold', ha='center', va='center')

# Target node (before update)
circle_t = plt.Circle(target_pos, 0.4, facecolor='#555555', edgecolor='black',
                       linewidth=2, zorder=3)
ax.add_patch(circle_t)
ax.text(target_pos[0], target_pos[1], '$v_i$', ha='center', va='center',
        fontsize=14, fontweight='bold', color='white', zorder=4)

ax.text(1.0, 7.5, '1. Message', fontsize=13, fontweight='bold', color=COLORS['blue'])

# --- Middle: aggregation box ---
agg_x, agg_y = 7.0, 4.0
box_agg = mpatches.FancyBboxPatch((agg_x - 1.0, agg_y - 0.5), 2.0, 1.0,
                                   boxstyle='round,pad=0.15',
                                   facecolor='#D6E4F0', edgecolor=COLORS['blue'], linewidth=2)
ax.add_patch(box_agg)
ax.text(agg_x, agg_y, 'AGGREGATE\n$\\bigoplus$', ha='center', va='center',
        fontsize=11, fontweight='bold', color=COLORS['blue'])

ax.annotate('', xy=(agg_x - 1.0, agg_y), xytext=(target_pos[0] + 0.45, target_pos[1]),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(7.0, 7.5, '2. Aggregate', fontsize=13, fontweight='bold', color=COLORS['blue'])

# --- Right: update box ---
upd_x, upd_y = 10.0, 4.0
box_upd = mpatches.FancyBboxPatch((upd_x - 1.0, upd_y - 0.5), 2.0, 1.0,
                                   boxstyle='round,pad=0.15',
                                   facecolor='#E8F5E9', edgecolor=COLORS['green'], linewidth=2)
ax.add_patch(box_upd)
ax.text(upd_x, upd_y, 'UPDATE\n$\\phi$', ha='center', va='center',
        fontsize=11, fontweight='bold', color=COLORS['green'])

ax.annotate('', xy=(upd_x - 1.0, upd_y), xytext=(agg_x + 1.0, agg_y),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(10.0, 7.5, '3. Update', fontsize=13, fontweight='bold', color=COLORS['green'])

# Updated node
upd_node_x = 12.5
circle_upd = plt.Circle((upd_node_x, upd_y), 0.4, facecolor=COLORS['green'],
                          edgecolor='black', linewidth=2, zorder=3)
ax.add_patch(circle_upd)
ax.text(upd_node_x, upd_y, "$v_i'$", ha='center', va='center',
        fontsize=14, fontweight='bold', color='white', zorder=4)

ax.annotate('', xy=(upd_node_x - 0.45, upd_y), xytext=(upd_x + 1.0, upd_y),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2.5))

# Self-connection from target to update
ax.annotate('', xy=(upd_x - 0.5, upd_y - 0.55), xytext=(target_pos[0] + 0.3, target_pos[1] - 0.4),
            arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5, linestyle='--'))
ax.text(7.5, 2.8, '$h_i^{(l)}$', fontsize=10, color='#555555', fontweight='bold')

ax.set_title('GNN Message Passing Framework', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig13_2_message_passing_gnn')
plt.show()

## Figure 13.3 -- Graph Pooling

Graph-level readout: individual node embeddings are aggregated via a pooling operation (sum, mean, or max) to produce a single global graph embedding vector.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 7)
ax.axis('off')

# --- Left: graph with node embeddings ---
g_pos = {0: (1.0, 5.0), 1: (0.0, 3.0), 2: (2.0, 3.0),
         3: (0.5, 1.0), 4: (1.5, 1.0)}
g_edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
g_colors = [COLORS['blue'], COLORS['red'], COLORS['green'],
            COLORS['amber'], COLORS['blue']]
g_labels = ['$h_1$', '$h_2$', '$h_3$', '$h_4$', '$h_5$']

# Draw edges
for i, j in g_edges:
    xi, yi = g_pos[i]
    xj, yj = g_pos[j]
    ax.plot([xi, xj], [yi, yj], color='#AAAAAA', linewidth=1.5, zorder=1)

# Draw nodes
for idx in range(5):
    x, y = g_pos[idx]
    circle = plt.Circle((x, y), 0.3, facecolor=g_colors[idx],
                         edgecolor='black', linewidth=1.5, zorder=3, alpha=0.85)
    ax.add_patch(circle)
    ax.text(x, y, g_labels[idx], ha='center', va='center',
            fontsize=12, fontweight='bold', color='white', zorder=4)

ax.text(1.0, 6.0, 'Node Embeddings', ha='center', fontsize=12,
        fontweight='bold', color=COLORS['blue'])

# --- Middle: embedding vectors flowing into pooling ---
# Draw embedding bars for each node
bar_x = 4.5
bar_w = 2.5
np.random.seed(13)
for idx in range(5):
    y_bar = 5.0 - idx * 1.0
    embedding = np.random.rand(6)
    for k, val in enumerate(embedding):
        ax.barh(y_bar, val * 0.35, left=bar_x + k * 0.4, height=0.5,
                color=g_colors[idx], alpha=0.6, edgecolor='none')
    # Arrow from node to bar
    nx, ny = g_pos[idx]
    ax.annotate('', xy=(bar_x, y_bar), xytext=(nx + 0.35, ny),
                arrowprops=dict(arrowstyle='->', color=g_colors[idx],
                                lw=1.2, alpha=0.5))

ax.text(bar_x + 1.2, 6.0, 'Embedding Vectors', ha='center', fontsize=12,
        fontweight='bold', color='#555555')

# --- Pooling box ---
pool_x, pool_y = 9.0, 3.0
box_pool = mpatches.FancyBboxPatch((pool_x - 1.2, pool_y - 0.7), 2.4, 1.4,
                                    boxstyle='round,pad=0.15',
                                    facecolor='#FFF3E0', edgecolor=COLORS['amber'], linewidth=2.5)
ax.add_patch(box_pool)
ax.text(pool_x, pool_y + 0.15, 'READOUT', ha='center', va='center',
        fontsize=12, fontweight='bold', color=COLORS['amber'])
ax.text(pool_x, pool_y - 0.25, '(sum / mean / max)', ha='center', va='center',
        fontsize=8, color=COLORS['amber'])

# Arrows from bars to pooling
for idx in range(5):
    y_bar = 5.0 - idx * 1.0
    ax.annotate('', xy=(pool_x - 1.2, pool_y),
                xytext=(bar_x + bar_w, y_bar),
                arrowprops=dict(arrowstyle='->', color='#888888', lw=1.0, alpha=0.5))

# --- Right: global graph embedding ---
out_x, out_y = 12.5, 3.0
box_out = mpatches.FancyBboxPatch((out_x - 1.3, out_y - 0.7), 2.6, 1.4,
                                   boxstyle='round,pad=0.15',
                                   facecolor='#D6E4F0', edgecolor=COLORS['blue'], linewidth=2.5)
ax.add_patch(box_out)
ax.text(out_x, out_y + 0.15, 'Graph\nEmbedding $h_G$', ha='center', va='center',
        fontsize=12, fontweight='bold', color=COLORS['blue'])

ax.annotate('', xy=(out_x - 1.3, out_y), xytext=(pool_x + 1.2, pool_y),
            arrowprops=dict(arrowstyle='->', color='black', lw=2.5))

ax.set_title('Graph-Level Readout (Pooling)', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig13_3_graph_pooling')
plt.show()

## Figure 13.4 -- Spectral Graph Analysis

Graph Laplacian eigenvalues and eigenvectors: a small graph is shown with node colors representing the values of the first few non-trivial eigenvectors of the graph Laplacian.

In [ ]:
# Build a small graph and compute graph Laplacian
node_pos_sp = {0: (0, 2), 1: (1, 3), 2: (2, 2), 3: (1, 1),
               4: (3, 3), 5: (4, 2), 6: (3, 1), 7: (2, 0)}
edges_sp = [(0, 1), (0, 3), (1, 2), (1, 4), (2, 3), (2, 4), (2, 6),
            (3, 7), (4, 5), (5, 6), (6, 7)]
n_sp = len(node_pos_sp)

# Adjacency and Laplacian
A_sp = np.zeros((n_sp, n_sp))
for i, j in edges_sp:
    A_sp[i, j] = 1
    A_sp[j, i] = 1
D_sp = np.diag(A_sp.sum(axis=1))
L_sp = D_sp - A_sp

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(L_sp)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

# Panel 0: eigenvalue spectrum
ax0 = axes[0]
ax0.bar(range(n_sp), eigenvalues, color=COLORS['blue'], alpha=0.8, edgecolor='black', linewidth=0.8)
ax0.set_xlabel('Index $k$', fontsize=11)
ax0.set_ylabel('$\\lambda_k$', fontsize=11)
ax0.set_title('Laplacian Eigenvalues', fontweight='bold', fontsize=12)
ax0.set_xticks(range(n_sp))

# Panels 1-3: eigenvectors 1, 2, 3 (skip eigenvector 0 which is constant)
evec_indices = [1, 2, 3]
evec_titles = ['Fiedler Vector ($\\phi_1$)', 'Eigenvector $\\phi_2$', 'Eigenvector $\\phi_3$']

for panel, (ev_idx, title) in enumerate(zip(evec_indices, evec_titles)):
    ax = axes[panel + 1]
    ax.set_xlim(-0.8, 4.8)
    ax.set_ylim(-0.8, 3.8)
    ax.axis('off')
    ax.set_title(f'{title}\n$\\lambda = {eigenvalues[ev_idx]:.2f}$',
                 fontweight='bold', fontsize=11)

    evec = eigenvectors[:, ev_idx]
    vmin, vmax = evec.min(), evec.max()

    # Draw edges
    for i, j in edges_sp:
        xi, yi = node_pos_sp[i]
        xj, yj = node_pos_sp[j]
        ax.plot([xi, xj], [yi, yj], color='#BBBBBB', linewidth=1.2, zorder=1)

    # Draw nodes colored by eigenvector value
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable, RdBu_r
    norm = Normalize(vmin=-max(abs(vmin), abs(vmax)), vmax=max(abs(vmin), abs(vmax)))
    cmap = plt.cm.RdBu_r

    for idx in range(n_sp):
        x, y = node_pos_sp[idx]
        color = cmap(norm(evec[idx]))
        circle = plt.Circle((x, y), 0.28, facecolor=color,
                             edgecolor='black', linewidth=1.5, zorder=3)
        ax.add_patch(circle)
        txt_color = 'white' if abs(evec[idx]) > 0.3 * max(abs(vmin), abs(vmax)) else 'black'
        ax.text(x, y, f'{evec[idx]:.2f}', ha='center', va='center',
                fontsize=7, fontweight='bold', color=txt_color, zorder=4)

    # Add colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6, aspect=15, pad=0.02)
    cbar.set_label('Value', fontsize=8)
    cbar.ax.tick_params(labelsize=7)

fig.suptitle('Spectral Analysis of the Graph Laplacian', fontweight='bold', fontsize=15, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig13_4_spectral_graph')
plt.show()

## Summary

- **fig13_1**: Graph with node features and adjacency matrix
- **fig13_2**: GNN message passing framework (message, aggregate, update)
- **fig13_3**: Graph-level readout via pooling to global embedding
- **fig13_4**: Spectral graph Laplacian analysis with eigenvector node coloring